# M7 TNT-inspired edit benchmark — Gold-dev route

This benchmark replaces autoregressive sentence generation with parallel character-edit prediction. For each source position, the encoder predicts `KEEP`, `DELETE`, `SUBSTITUTE`, or `EXPAND`, plus a bounded replacement segment. Reconstruction is deterministic, so repeated decoding loops and EOS failures are impossible.

As requested, no synthetic-dev split is created. Synthetic pretraining and subsequent Gold fine-tuning are monitored/selected on Gold-dev. Gold-test remains frozen until every condition is final. This reuse of Gold-dev must be reported as a limitation.

In [1]:
from pathlib import Path
import json
import math
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.modeling import tnt_benchmark_runner as benchmark

benchmark.configure_benchmark('tnt_edit_gold_dev_v4')
pd.set_option('display.max_colwidth', 140)
print('Benchmark:', benchmark.BENCHMARK_NAME)
print('Results:', benchmark.RESULTS_DIR.resolve())

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Benchmark: tnt_edit_gold_dev_v4
Results: C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4


## 1. Fixed protocol

In [2]:
PROTOCOL = benchmark.CompactBenchmarkProtocol(
    seed=2026,
    matched_unique_rows=3330,
    synthetic_epochs=20.0,  # 1,060 optimizer steps; still below TNT's published 64 epochs
    gold_epochs=20.0,
    synthetic_learning_rate=3e-4,
    gold_learning_rate=1e-4,
    synthetic_batch_size=64,
    gold_batch_size=32,
    evaluation_batch_size=64,  # Gold-dev has only 30 rows: one parallel batch
    generated_eval_interval_epochs=5,
)
PROTOCOL.validate()
print(PROTOCOL)
print('Synthetic optimizer steps:', math.ceil(3330 / PROTOCOL.synthetic_batch_size) * int(PROTOCOL.synthetic_epochs))

CompactBenchmarkProtocol(seed=2026, matched_unique_rows=3330, synthetic_epochs=20.0, gold_epochs=20.0, synthetic_learning_rate=0.0003, gold_learning_rate=0.0001, synthetic_batch_size=64, gold_batch_size=32, evaluation_batch_size=64, generated_eval_interval_epochs=5)
Synthetic optimizer steps: 1060


## 2. Load the original matched 3,330 rows per method

In [3]:
gold, method_training, dataset_audit = benchmark.prepare_benchmark_data(PROTOCOL)
print('Gold:', {split: len(frame) for split, frame in gold.items()})
print('Synthetic:', {method: len(frame) for method, frame in method_training.items()})
display(dataset_audit)

Gold: {'train': 142, 'dev': 30, 'test': 29}
Synthetic: {'m1': 3330, 'm2': 3330, 'm3': 3330, 'm4': 3330, 'm6': 3330}


,method,loaded_rows,unique_pair_rows,unique_target_rows,selected_rows,common_target_pool,seed,dev_input_overlap,dev_pair_overlap,test_input_overlap,test_pair_overlap,source_over_255_chars,target_over_255_chars
0,m1,17777,16377,16375,3330,3430,2026,0,0,0,0,8,6
1,m2,17777,16544,16375,3330,3430,2026,0,0,0,0,10,6
2,m3,17777,16384,16375,3330,3430,2026,0,0,0,0,8,6
3,m4,3433,3433,3430,3330,3430,2026,0,0,0,0,7,6
4,m6,17777,16427,16375,3330,3430,2026,0,0,0,0,6,6


## 3. Create/reuse the shared TNT initialization

In [4]:
shared_initialization = benchmark.prepare_shared_initialization(
    gold, method_training, PROTOCOL, output_slots_per_source=32
)
manifest = json.loads((shared_initialization / 'initialization_manifest.json').read_text(encoding='utf-8'))
display(manifest)
print('Shared model:', shared_initialization)
print('Parameters:', f"{manifest['parameter_count']:,}")

[M7] Reusing TNT edit initialization: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


{'created_at_utc': '2026-09-12T15:45:22.263772+00:00',
 'fingerprint': 'eb64e8c2d3de53fd1a772d766bc9a97ba109c8c97730f8c6ec037444a1b8a804',
 'implementation': 'TNT-inspired non-autoregressive edit Transformer',
 'paper': 'Tan et al. (2020), TNT',
 'segmentation': 'character-level',
 'operations': ['keep', 'delete', 'substitute', 'expand'],
 'architecture': {'d_model': 192,
  'attention_heads': 4,
  'encoder_layers': 3,
  'feed_forward_size': 768,
  'dropout': 0.1,
  'max_position_embeddings': 256,
  'output_slots_per_source': 32,
  'operation_loss_weight': 1.0,
  'character_loss_weight': 1.0,
  'label_smoothing': 0.1},
 'seed': 2026,
 'parameter_count': 2148676,
 'training_text_count_used_for_vocabulary': 33584}

Shared model: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de
Parameters: 2,148,676


## 4. Optional operation-label preview

This cell performs no training. It shows how one pair becomes source-anchored operation labels.

In [5]:
from transformers import AutoTokenizer
from src.modeling.tnt_edit_transformer import OPERATION_NAMES, tnt_edit_annotations

tokenizer = AutoTokenizer.from_pretrained(shared_initialization)
example = method_training['m1'].iloc[0]
source_ids = tokenizer(example.source_text)['input_ids']
target_ids = tokenizer(text_target=example.target_text)['input_ids']
operations, segments = tnt_edit_annotations(
    source_ids, target_ids, eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id, output_slots_per_source=32
)
preview = pd.DataFrame({
    'source_character': tokenizer.convert_ids_to_tokens(source_ids),
    'operation': [OPERATION_NAMES[value] for value in operations],
    'predicted_segment_target': [tokenizer.decode([value for value in segment if value >= 0 and value != tokenizer.pad_token_id]) for segment in segments],
})
display(example[['source_text', 'target_text']])
display(preview[preview.operation != 'keep'].head(40))

You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


source_text    * 1 kasou fariné bou meune dem fépp
target_text       * 1 kaasu farine bu mën dem fépp
Name: 0, dtype: str

,source_character,operation,predicted_segment_target
6,s,substitute,a
7,o,substitute,s
15,é,substitute,e
18,o,delete,
22,e,delete,
23,u,substitute,ë
25,e,delete,


## 6. M1 — Beqi-inspired rules

## 5. Gold-only control

In [ ]:
RUN_GOLD = False
RERUN_GOLD = False
if RUN_GOLD:
    gold_selection = benchmark.run_gold_control(gold, shared_initialization, PROTOCOL, rerun=RERUN_GOLD)
    display(pd.DataFrame(gold_selection['candidate_runs']))
else:
    print('Gold disabled. Set RUN_GOLD=True when ready.')

Gold disabled. Set RUN_GOLD=True when ready.


In [7]:
RUN_M1 = True
RERUN_M1 = False
if RUN_M1:
    m1_selection = benchmark.run_synthetic_method('m1', method_training['m1'], gold, shared_initialization, PROTOCOL, rerun=RERUN_M1)
    display(pd.DataFrame(m1_selection['candidate_runs']))
else:
    print('M1 disabled. Set RUN_M1=True when ready.')


[M7] START m1 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

[M7] Loaded 2,148,676 total parameters; training 2,148,676 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


Tokenizing TNT edit pairs:   0%|          | 0/3330 [00:00<?, ? examples/s]

...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntEditTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntEditTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.1810730253353204
[compact benchmark] m1: continuous 20-epoch gold run complete; selected dev CER=0.1810730253353204


,regime,attempt,checkpoint,development_metrics
0,gold_finetune_continuous,1,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model,"{'rows': 30, 'cer': 0.1810730253353204, 'wer': 0.6677966101694915, 'chrf': 55.88812485988872, 'exact_match': 0.0, 'correction_tp': 2, 'c..."


## 7. M2 — Eflomal extension

In [14]:
RUN_M2 = True
RERUN_M2 = False
if RUN_M2:
    m2_selection = benchmark.run_synthetic_method('m2', method_training['m2'], gold, shared_initialization, PROTOCOL, rerun=RERUN_M2)
    display(pd.DataFrame(m2_selection['candidate_runs']))
else:
    print('M2 disabled. Set RUN_M2=True when ready.')


[M7] START m2 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntEditTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "character_loss_weight": 1.0,
  "d_model": 192,
  "decoder_start_token_id": 1,
  "dropout": 0.1,
  "dtype": "float32",
  "encod

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}



[M7] Loaded 2,148,676 total parameters; training 2,148,676 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntEditTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntEditTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m2 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.19672131147540983
[compact benchmark] m2: continuous 20-epoch gold run complete; selected dev CER=0.19672131147540983


,regime,attempt,checkpoint,development_metrics
0,gold_finetune_continuous,1,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\gold_finetune_continuous\attempt_001\best_model,"{'rows': 30, 'cer': 0.19672131147540983, 'wer': 0.6576271186440678, 'chrf': 56.4555618836903, 'exact_match': 0.0, 'correction_tp': 2, 'c..."


## 5. Gold-only control

In [15]:
RUN_GOLD = True
RERUN_GOLD = False
if RUN_GOLD:
    gold_selection = benchmark.run_gold_control(gold, shared_initialization, PROTOCOL, rerun=RERUN_GOLD)
    display(pd.DataFrame(gold_selection['candidate_runs']))
else:
    print('Gold disabled. Set RUN_GOLD=True when ready.')


[M7] START gold / gold_finetune_continuous / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntEditTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "character_loss_weight": 1.0,
  "d_model": 192,
  "decoder_start_token_id": 1,
  "dropout": 0.1,
  "dtype": "float32",
  "encod

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}



[M7] Loaded 2,148,676 total parameters; training 2,148,676 (100.0000%) via full_finetune
[M7] Tokenizing 142 training and 30 development pairs


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntEditTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntEditTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE gold / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.23025335320417287
[compact benchmark] gold: continuous 20-epoch gold run complete; selected dev CER=0.23025335320417287


,regime,attempt,checkpoint,development_metrics
0,gold_finetune_continuous,1,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_m...,"{'rows': 30, 'cer': 0.23025335320417287, 'wer': 0.6271186440677966, 'chrf': 56.07386560171533, 'exact_match': 0.0, 'correction_tp': 1, '..."


## 8. M3 — CMDR extension

In [16]:
RUN_M3 = True
RERUN_M3 = False
if RUN_M3:
    m3_selection = benchmark.run_synthetic_method('m3', method_training['m3'], gold, shared_initialization, PROTOCOL, rerun=RERUN_M3)
    display(pd.DataFrame(m3_selection['candidate_runs']))
else:
    print('M3 disabled. Set RUN_M3=True when ready.')


[M7] START m3 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntEditTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "character_loss_weight": 1.0,
  "d_model": 192,
  "decoder_start_token_id": 1,
  "dropout": 0.1,
  "dtype": "float32",
  "encod

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}



[M7] Loaded 2,148,676 total parameters; training 2,148,676 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntEditTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntEditTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m3 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.19225037257824143
[compact benchmark] m3: continuous 20-epoch gold run complete; selected dev CER=0.19225037257824143


,regime,attempt,checkpoint,development_metrics
0,gold_finetune_continuous,1,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\gold_finetune_continuous\attempt_001\best_model,"{'rows': 30, 'cer': 0.19225037257824143, 'wer': 0.6677966101694915, 'chrf': 55.282623261035724, 'exact_match': 0.0, 'correction_tp': 2, ..."


## 9. M4 — Oolel external LLM-generated data

In [17]:
RUN_M4 = True
RERUN_M4 = False
if RUN_M4:
    m4_selection = benchmark.run_synthetic_method('m4', method_training['m4'], gold, shared_initialization, PROTOCOL, rerun=RERUN_M4)
    display(pd.DataFrame(m4_selection['candidate_runs']))
else:
    print('M4 disabled. Set RUN_M4=True when ready.')


[M7] START m4 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntEditTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "character_loss_weight": 1.0,
  "d_model": 192,
  "decoder_start_token_id": 1,
  "dropout": 0.1,
  "dtype": "float32",
  "encod

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}



[M7] Loaded 2,148,676 total parameters; training 2,148,676 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntEditTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntEditTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m4 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.19299552906110284
[compact benchmark] m4: continuous 20-epoch gold run complete; selected dev CER=0.19299552906110284


,regime,attempt,checkpoint,development_metrics
0,gold_finetune_continuous,1,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model,"{'rows': 30, 'cer': 0.19299552906110284, 'wer': 0.6474576271186441, 'chrf': 56.59416354050008, 'exact_match': 0.0, 'correction_tp': 2, '..."


## 10. M6 — Gold-informed linguistic generator

In [18]:
RUN_M6 = True
RERUN_M6 = False
if RUN_M6:
    m6_selection = benchmark.run_synthetic_method('m6', method_training['m6'], gold, shared_initialization, PROTOCOL, rerun=RERUN_M6)
    display(pd.DataFrame(m6_selection['candidate_runs']))
else:
    print('M6 disabled. Set RUN_M6=True when ready.')


[M7] START m6 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\config.json
You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntEditTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "character_loss_weight": 1.0,
  "d_model": 192,
  "decoder_start_token_id": 1,
  "dropout": 0.1,
  "dtype": "float32",
  "encod

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\shared_initialization\tnt_edit_char_eb64e8c2d3de\generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}



[M7] Loaded 2,148,676 total parameters; training 2,148,676 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntEditTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntEditTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m6 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.19374068554396423
[compact benchmark] m6: continuous 20-epoch gold run complete; selected dev CER=0.19374068554396423


,regime,attempt,checkpoint,development_metrics
0,gold_finetune_continuous,1,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model,"{'rows': 30, 'cer': 0.19374068554396423, 'wer': 0.6372881355932203, 'chrf': 56.596508918625766, 'exact_match': 0.0, 'correction_tp': 3, ..."


## 11. Gold-dev summary

In [19]:
development_summary = benchmark.development_summary(gold['dev'])
display(development_summary)
print('Missing final selections:', benchmark.missing_selections())
benchmark.plot_summary(development_summary, benchmark.RESULTS_DIR / 'development_metrics.png', 'TNT edit benchmark — Gold-dev')

,condition,kind,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate,checkpoint
0,m0_identity,reference_baseline,30,0.233234,0.630508,56.052143,0.0,0,0,72,0.000000,0.000000,0.000000,0.000000,0,0.0,NaN
1,gold,gold_control,30,0.230253,0.627119,56.073866,0.0,1,1,71,0.500000,0.013889,0.027027,0.500000,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_m...
2,m1,synthetic_then_gold,30,0.181073,0.667797,55.888125,0.0,2,67,70,0.028986,0.027778,0.028369,0.971014,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model
3,m2,synthetic_then_gold,30,0.196721,0.657627,56.455562,0.0,2,60,70,0.032258,0.027778,0.029851,0.967742,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\gold_finetune_continuous\attempt_001\best_model
4,m3,synthetic_then_gold,30,0.192250,0.667797,55.282623,0.0,2,72,70,0.027027,0.027778,0.027397,0.972973,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\gold_finetune_continuous\attempt_001\best_model
5,m4,synthetic_then_gold,30,0.192996,0.647458,56.594164,0.0,2,60,70,0.032258,0.027778,0.029851,0.967742,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model
6,m6,synthetic_then_gold,30,0.193741,0.637288,56.596509,0.0,3,68,69,0.042254,0.041667,0.041958,0.957746,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model


Missing final selections: []


WindowsPath('C:/Users/thioy/Desktop/School/PFE/results/m7_seq2seq/tnt_edit_gold_dev_v4/development_metrics.png')

## 12. Frozen Gold-test — run once only after every condition is accepted

In [21]:
CONFIRM_FROZEN_TEST = True
if CONFIRM_FROZEN_TEST:
    frozen_test_summary = benchmark.evaluate_frozen_test(gold['test'], PROTOCOL, confirm=True)
    display(frozen_test_summary)
else:
    print('Frozen Gold-test remains disabled.')

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_model\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_model\config.json
You are using a model of type `wolof_tnt_edit_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntEditTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "character_loss_weight": 1.0,
  "d_model": 192,
  "decoder_start_token_id": 1,
  "dropout": 0.1,
  

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_model\generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": [
    2
  ],
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": [
    2
  ],
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_d

...

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model\generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": [
    2
  ],
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_d

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model\generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": [
    2
  ],
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_d

,condition,kind,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate,checkpoint,predictions_path,inference_seconds
0,m0_identity,reference_baseline,29,0.220328,0.570093,58.495661,0.000000,0,0,75,0.000000,0.000000,0.000000,0.000000,0,0.0,NaN,NaN,NaN
1,gold,gold_control,29,0.220328,0.570093,58.495661,0.000000,0,0,75,0.000000,0.000000,0.000000,0.000000,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\gold\seed_2026\gold_finetune_continuous\attempt_001\best_m...,C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4\frozen_test\gold\test_predictions.csv,0.119836
2,m1,synthetic_then_gold,29,0.195410,0.607477,56.728536,0.034483,2,66,73,0.029412,0.026667,0.027972,0.970588,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model,C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4\frozen_test\m1\test_predictions.csv,0.101182
3,m2,synthetic_then_gold,29,0.200000,0.613707,57.903232,0.000000,0,59,75,0.000000,0.000000,0.000000,1.000000,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m2\seed_2026\gold_finetune_continuous\attempt_001\best_model,C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4\frozen_test\m2\test_predictions.csv,0.089170
4,m3,synthetic_then_gold,29,0.209836,0.629283,55.091236,0.000000,1,67,74,0.014706,0.013333,0.013986,0.985294,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m3\seed_2026\gold_finetune_continuous\attempt_001\best_model,C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4\frozen_test\m3\test_predictions.csv,0.098846
5,m4,synthetic_then_gold,29,0.191475,0.598131,58.789204,0.000000,1,57,74,0.017241,0.013333,0.015038,0.982759,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m4\seed_2026\gold_finetune_continuous\attempt_001\best_model,C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4\frozen_test\m4\test_predictions.csv,0.101249
6,m6,synthetic_then_gold,29,0.205902,0.591900,57.982033,0.000000,2,55,73,0.035088,0.026667,0.030303,0.964912,0,0.0,C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_edit_gold_dev_v4\m6\seed_2026\gold_finetune_continuous\attempt_001\best_model,C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_edit_gold_dev_v4\frozen_test\m6\test_predictions.csv,0.092982
